# Baseline Modelling

Tujuan: Notebook ini bertujuan membangun **baseline classification model** untuk memprediksi hasil pertandingan sepak bola internasional menggunakan fitur-fitur dasar yang telah dihasilkan pada tahap *Feature Engineering*. Model baseline ini akan menjadi acuan utama sebelum dilakukan pengembangan fitur historis seperti performa lima pertandingan terakhir, rata-rata gol, hingga Elo Rating pada fase berikutnya.

## 1. Loading

Memuat dataset `baseline_features` yang telah disimpan pada tahap sebelumnya. 

In [10]:
import pandas as pd

In [11]:
df = pd.read_csv('../data/processed/baseline_features.csv')

In [12]:
df.head()

,date,home_team,away_team,tournament,neutral,match_result
0,1872-11-30,Scotland,England,Friendly,False,D
1,1873-03-08,England,Scotland,Friendly,False,H
2,1874-03-07,Scotland,England,Friendly,False,H
3,1875-03-06,England,Scotland,Friendly,False,D
4,1876-03-04,Scotland,England,Friendly,False,H


In [13]:
df.shape

(49433, 6)

Data sudah berhasil di load kembali dan siap digunakan.

## 2. Feature & Target Separation

Dataset yang telah dipersiapkan pada tahap sebelumnya
berisi fitur-fitur yang tersedia sebelum pertandingan dimulai
serta satu variabel target (`match_result`).

Pada tahap ini dataset dipisahkan menjadi:

- Feature (`X`)
- Target (`y`)

In [14]:
X = df.drop(columns=["match_result"])

y = df["match_result"]

## 3. Time-based Train-Test Split

 Sebelum membangun model, dataset perlu dibagi menjadi data latih (*training set*) dan data uji (*testing set*). Berbeda dengan sebagian besar dataset tabular yang menggunakan pembagian secara acak (*random split*), penelitian ini menggunakan *time-based split*. Pendekatan ini dipilih agar model hanya mempelajari pertandingan yang terjadi di masa lalu dan dievaluasi menggunakan pertandingan yang terjadi setelahnya. Dengan demikian, proses evaluasi lebih merepresentasikan kondisi nyata serta membantu mengurangi risiko *data leakage* akibat penggunaan informasi dari masa depan.

### 3.1 Menentukan Urutan

Kita harus memastikan dataset sudah urut berdasarkan tanggal.

In [15]:
df["date"].is_monotonic_increasing

False

Ternyata dataset belum urut. Kita akan urutkan terlebih dahulu berdasarkan tanggal.

In [16]:
df = df.sort_values("date").reset_index(drop=True)

In [17]:
df["date"].is_monotonic_increasing

True

Dataset berhasil di urutkan.

### 3.2 Split Dataset

Keputusannya, data akan di split 80:20 berdasarkan urutan tanggal.

In [18]:
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

### 3.3 Pisahkan Feature dan Target

Kita pisahkan fitur dan target untuk data train dan test.

In [19]:
# TRAIN
X_train = train_df.drop(columns=["match_result"])
y_train = train_df["match_result"]

# TEST
X_test = test_df.drop(columns=["match_result"])
y_test = test_df["match_result"]

### 3.4 Verifikasi
Memastikan proses berjalan lancar.

In [20]:
print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

Train : (39546, 6)
Test  : (9887, 6)


In [21]:
print(train_df["date"].min(), "-", train_df["date"].max())
print(test_df["date"].min(), "-", test_df["date"].max())

1872-11-30 - 2016-03-25
2016-03-25 - 2026-06-18



### 3.5 Insight

Dataset berhasil dibagi menggunakan pendekatan *time-based split* dengan proporsi
80% data latih dan 20% data uji berdasarkan urutan kronologis pertandingan.
Pendekatan ini memastikan model hanya mempelajari informasi dari masa lalu dan
dievaluasi menggunakan pertandingan yang terjadi setelahnya, sehingga proses
evaluasi menjadi lebih realistis serta membantu meminimalkan risiko *data leakage*.